<a href="https://colab.research.google.com/github/jsmak/mytransformer/blob/main/notebooks/10_Decoder.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### **디코더**
지금까지 구현해 온 각 클래스를 사용해, 「Decoder」를 구현합니다.

#### **(1) ScaledDotProductAttention 클래스**
「Scaled Dot-Product Attention」의 클래스입니다.

In [2]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

class ScaledDotProductAttention(nn.Module):
    def __init__(self, d_head):
        super().__init__()
        self.d_head = d_head

    def forward(self, Q, K, V, attn_mask=None):
        attention_scores = Q @ K.transpose(-2, -1)/ np.sqrt(self.d_head)

        if attn_mask is not None:
            attention_scores = attention_scores.masked_fill(attn_mask == 0, float("-inf"))

        attention_weights = F.softmax(attention_scores, dim=-1)
        context_vector = attention_weights @ V
        return context_vector

#### **(2)MultiHeadAttention 클래스**  
Multi-Head Attention」클래스입니다.

In [3]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads):
        super().__init__()

        assert d_model % num_heads == 0, "d_model must be divisible by num_heads"

        self.d_model = d_model
        self.num_heads = num_heads
        self.d_head = d_model // num_heads

        self.W_q = nn.Linear(d_model, d_model, bias=False)
        self.W_k = nn.Linear(d_model, d_model, bias=False)
        self.W_v = nn.Linear(d_model, d_model, bias=False)
        self.out = nn.Linear(d_model, d_model, bias=False)

        self.attention = ScaledDotProductAttention(self.d_head)

    def forward(self, xq, xk, xv, attn_mask=None):
        batch_size = xq.size(0)
        Q = self.W_q(xq).view(batch_size, -1, self.num_heads, self.d_head).transpose(1, 2)
        K = self.W_k(xk).view(batch_size, -1, self.num_heads, self.d_head).transpose(1, 2)
        V = self.W_v(xv).view(batch_size, -1, self.num_heads, self.d_head).transpose(1, 2)

        # Apply Scaled Dot Product Attention
        x = self.attention(Q, K, V, attn_mask)

        # Concatenate and apply final linear
        head_output = x.transpose(1, 2).contiguous().view(batch_size, -1, self.d_model)
        out = self.out(head_output)
        return out

#### **(3) Attention mask 생성**
Attention mask를 생성합니다..

In [4]:
def make_attention_mask(pad_mask=None, q_len=None, causal=False):
    attn_mask = None

    if pad_mask is not None:
        batch_size, k_len = pad_mask.shape

        if q_len is None:
            q_len = k_len
        attn_mask = pad_mask.unsqueeze(1).unsqueeze(2)
        attn_mask = attn_mask.expand(batch_size, 1, q_len, k_len)

    if causal:
        if q_len is None:
            raise ValueError("causal=True일 때는 q_len이 필요합니다.")

        k_len = q_len
        causal_mask = torch.tril(torch.ones(q_len, k_len))

        # [1, 1, q_len, k_len]
        causal_mask = causal_mask.unsqueeze(0).unsqueeze(0)

        if attn_mask is None:
            attn_mask = causal_mask
        else:
            attn_mask = attn_mask * causal_mask
    return attn_mask

**(4) DecoderLayer 클래스**  
「Attention Is All You Need」논문의, 1회 Decoder동작을 처리를 수행하는 DecoderLayer 클래스입니다.

In [5]:
class DecoderLayer(nn.Module):
    def __init__(self, d_model, num_heads):
        super().__init__()

        self.self_attn = MultiHeadAttention(d_model, num_heads)
        self.cross_attn = MultiHeadAttention(d_model, num_heads)

        d_ff = d_model * 4

        self.W_ffn1 = nn.Parameter(torch.randn(d_model, d_ff))
        self.b_ffn1 = nn.Parameter(torch.randn(d_ff))
        self.W_ffn2 = nn.Parameter(torch.randn(d_ff, d_model))
        self.b_ffn2 = nn.Parameter(torch.randn(d_model))

        # Decoder는 Add & Norm이 3개
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.norm3 = nn.LayerNorm(d_model)

    def forward(self, x, encoder_out, tgt_self_mask=None, cross_mask=None):
        # 1. Masked Multi-Head Self-Attention
        self_attn_out = self.self_attn(x, x, x, attn_mask=tgt_self_mask)
        norm1_out = self.norm1(x + self_attn_out)

        # 2. Encoder-Decoder Attention, Cross Attention
        cross_attn_out = self.cross_attn(norm1_out, encoder_out, encoder_out, attn_mask=cross_mask)
        norm2_out = self.norm2(norm1_out + cross_attn_out)

        # 3. Feed Forward Network
        f_out = F.relu(norm2_out @ self.W_ffn1 + self.b_ffn1)
        f_out = f_out @ self.W_ffn2 + self.b_ffn2

        # 4. Add & Norm
        final_out = self.norm3(norm2_out + f_out)
        return final_out

**(5) Decoder 클래스**  
「Attention Is All You Need」 논문의 DecoderLayer 클래스를 반복하는 Decoder 클래스입니다.

In [6]:
class Decoder(nn.Module):
    def __init__(self, num_layers, d_model, num_heads):
        super().__init__()

        self.layers = nn.ModuleList([
            DecoderLayer(d_model, num_heads)
            for _ in range(num_layers)
        ])

    def forward(self, x, encoder_out, tgt_pad_mask=None, src_pad_mask=None):
        tgt_len = x.size(1)
        tgt_self_mask = make_attention_mask(pad_mask=tgt_pad_mask, q_len=tgt_len, causal=True)
        cross_mask = make_attention_mask(pad_mask=src_pad_mask, q_len=tgt_len, causal=False )

        for layer in self.layers:
            x = layer(x, encoder_out, tgt_self_mask=tgt_self_mask, cross_mask=cross_mask)

        # padding 위치 출력을 0으로 유지하고 싶을 경우
        if tgt_pad_mask is not None:
            x = x * tgt_pad_mask.unsqueeze(-1)
        return x

**(6) Decoder 클래스 실행**   
Decoder 클래스를 실행합니다.

In [7]:
# 파라미터 설정
d_model = 512  # 입력 차원의 수
n_head = 8     # 어텐션 헤드 수
num_layers = 6 # 인코더 레이어 수
batch_size = 32 # 배치 크기
tgt_seq_len = 100   # 입력 시퀀스의 길이
src_seq_len = 120

# 인코더 인스턴스화
decoder = Decoder(num_layers, d_model, n_head)

# 모델을 평가 모드로 설정
decoder.eval()

# 더미 입력 데이터
decode_input_tensor = torch.rand(batch_size, tgt_seq_len, d_model) # 디코더 입력 텐서
enc_output = torch.rand(batch_size, src_seq_len, d_model)   # 인코더의 출력 텐서


src_mask = torch.ones(batch_size, src_seq_len)
tgt_mask = torch.ones(batch_size, tgt_seq_len)

# 디코더에 데이터를 전달
with torch.no_grad(): # 기울기 계산하지 않음
  encoded_output = decoder(decode_input_tensor, enc_output, tgt_mask, src_mask)

print("Decoded input shape:", decode_input_tensor.shape )   # 입력 텐서의 shape: [batch_size, seq_len, d_model]
print("Decoded output shape:", encoded_output.shape) # 출력 텐서의 shape: [batch_size, seq_len, d_model]
print("Decoded output:", encoded_output)

Decoded input shape: torch.Size([32, 100, 512])
Decoded output shape: torch.Size([32, 100, 512])
Decoded output: tensor([[[ 2.7203e-01, -3.8313e-01,  1.0012e+00,  ...,  1.5498e-01,
           1.1344e-01,  2.0343e+00],
         [ 4.1013e-01, -1.4852e-01, -3.2251e-01,  ...,  8.8367e-02,
           3.8343e-01,  2.3309e+00],
         [ 2.8082e-01, -8.5819e-01,  8.3497e-01,  ...,  2.8159e-02,
           2.1779e-01,  1.5014e+00],
         ...,
         [ 2.6543e-01, -4.4282e-03, -3.7213e-01,  ...,  9.2499e-01,
          -3.1401e-01,  2.6649e+00],
         [ 5.4138e-01,  1.3943e-01, -5.6671e-01,  ...,  4.8146e-01,
          -4.3797e-01,  2.1672e+00],
         [ 6.7152e-01, -6.4099e-01,  5.6592e-01,  ...,  4.2869e-01,
          -4.7137e-01,  2.0522e+00]],

        [[-5.3636e-02, -5.5270e-01, -1.1382e+00,  ..., -1.7678e-01,
          -4.9519e-01,  1.5145e+00],
         [-2.8259e-01, -1.0506e-01,  3.5499e-01,  ..., -5.5954e-01,
          -3.0513e-01,  2.0028e+00],
         [ 2.3947e-01, -7.9544e